Célula de inspeção manual — só para conferência rápida ao rodar este
notebook. O cálculo oficial de `% de falha por regra` que alimenta a Gold
vive em `gold_dq_resumo.ipynb`, não aqui.


# Gold — dq_resumo_por_regra / dq_resumo_por_tabela
Squad 1 · Dupla 3

Lê `dq_monitoring_logs` e gera as duas tabelas Gold pedidas no card:
`squad1.gold_dq_resumo_por_regra` (% de falha por regra por dia) e
`squad1.gold_dq_resumo_por_tabela` (% de registros limpos por tabela por
hora). Gravadas em Delta e, depois de validar os números, também no SQL
Server pro Looker.

In [0]:
# Célula 1 — setup
import os
from dotenv import load_dotenv
from pyspark.sql import functions as F
import pandas as pd
from deltalake import DeltaTable, write_deltalake

load_dotenv(".env")

STORAGE_ACCOUNT = "internshipdatalake"
SQUAD_CONTAINER = "squad1"
lakehouse_base = f"abfss://{SQUAD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

storage_options = {
    "azure_storage_account_name": STORAGE_ACCOUNT,
    "azure_storage_client_id": os.getenv("ADLS_CLIENT_ID"),
    "azure_storage_client_secret": os.getenv("ADLS_CLIENT_SECRET"),
    "azure_storage_tenant_id": os.getenv("ADLS_TENANT_ID"),
}

dq_logs_path = f"{lakehouse_base}/dq_monitoring_logs"
silver_produtos_path = f"{lakehouse_base}/silver/ecommerce_produtos"
silver_categorias_path = f"{lakehouse_base}/silver/ecommerce_categorias"
gold_resumo_por_regra_path = f"{lakehouse_base}/gold/dq_resumo_por_regra"
gold_resumo_por_tabela_path = f"{lakehouse_base}/gold/dq_resumo_por_tabela"

def ler_delta_pandas(caminho):
    return DeltaTable(caminho, storage_options=storage_options).to_pandas()

print("Setup concluído.")

In [0]:
# Célula 2 — dq_resumo_por_regra: % de falha por regra, por dia
df_logs = ler_delta_pandas(dq_logs_path)
df_logs["_data"] = df_logs["arquivo_origem"].str.extract(r"(\d{4}/\d{2}/\d{2})")[0].str.replace("/", "-")

resumo_por_regra = (df_logs
    .groupby(["tabela", "regra", "_data"])
    .agg(qtd_falhas=("qtd_registros_falhos", "sum"), qtd_total=("qtd_registros_total", "sum"))
    .reset_index())
resumo_por_regra["pct_falha"] = (resumo_por_regra["qtd_falhas"] / resumo_por_regra["qtd_total"] * 100).round(2)

display(resumo_por_regra.sort_values("pct_falha", ascending=False))

In [0]:
# Célula 3 (substituindo a anterior) — dq_resumo_por_tabela: % de registros
# limpos por tabela, por hora. Vem da Silver, não do dq_monitoring_logs,
# porque "limpo" é por LINHA (passou em todas as regras aplicáveis), e o
# log só guarda contagem agregada por regra — não dá pra reconstruir isso
# a partir dele sem contar a mesma linha várias vezes.
def calcular_limpos(caminho_silver, nome_tabela):
    df = ler_delta_pandas(caminho_silver)
    colunas_regra = [c for c in df.columns if c.startswith("regra_")]

    # uma linha é "limpa" se nenhuma regra aplicável deu False
    # (comparar com None/NaN dá False automaticamente, então regra não
    # aplicável — tipo regra_09 nos itens inativos — não conta como falha)
    falhou_alguma = (df[colunas_regra] == False).any(axis=1)
    df["_limpo"] = ~falhou_alguma

    df["_data"] = df["bronze_source_file"].str.extract(r"(\d{4}/\d{2}/\d{2})")[0].str.replace("/", "-")
    df["_hora"] = df["bronze_source_file"].str.extract(r"/(\d{2})\d{4}/")[0]

    resumo = (df.groupby(["_data", "_hora"])
        .agg(qtd_limpos=("_limpo", "sum"), qtd_total=("_limpo", "size"))
        .reset_index())
    resumo["tabela"] = nome_tabela
    resumo["pct_limpo"] = (resumo["qtd_limpos"] / resumo["qtd_total"] * 100).round(2)
    return resumo

resumo_por_tabela = pd.concat([
    calcular_limpos(silver_produtos_path, "ecommerce_produtos"),
    calcular_limpos(silver_categorias_path, "ecommerce_categorias"),
], ignore_index=True)

display(resumo_por_tabela.sort_values(["tabela", "_data", "_hora"]))

In [0]:
# Célula 4 — gravação em Delta (confirma os números antes de rodar o SQL Server)
write_deltalake(gold_resumo_por_regra_path, resumo_por_regra, mode="overwrite", storage_options=storage_options)
write_deltalake(gold_resumo_por_tabela_path, resumo_por_tabela, mode="overwrite", storage_options=storage_options)
print("Gold gravado em Delta. Gravação no SQL Server fica para depois de validar.")